# BIU DS23 · Module 4 · The Pipeline Assignment · Starter Notebook
**Due August 30. Read the assignment document first, then work here.**

This notebook is scaffolding, not a solution. Your job is to build one leakage-safe
Pipeline, from raw data to model, and to JUSTIFY every decision. Remember the rubric:
**70% of the grade is on your reasoning, not on the score.** A small or even negative
lift with excellent justification beats a high score you cannot explain.

**Two tracks (pick one, see the assignment document):**
1. **Olist marketplace.** Beat the frozen benchmark you generated by running
   `DS23_Module4_Benchmark.ipynb` (it writes `module4_benchmark.json` to your Drive).
2. **Your own data.** The same process on your capstone dataset. You build your own
   minimal baseline first, then measure your lift against it.

**The one rule that never bends:** every step that learns from the data (imputation,
scaling, encoding, resampling) lives INSIDE the Pipeline, so it is fit on the training
fold only. Anything else is leakage, and leakage caps your grade.

## 0 · Setup

In [ ]:
!pip install -q scikit-learn pandas numpy imbalanced-learn category_encoders

import numpy as np, pandas as pd, json
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

from google.colab import drive
drive.mount("/content/drive")
DATA_PATH = "/content/drive/MyDrive/DS23/olist/"


## 1 · Load the raw tables (Olist track)
For the Olist track, load the raw tables. For your own-data track, load your dataset
instead and skip to section 3.

In [ ]:
items    = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
reviews  = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
orders   = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
print("tables loaded.")


## 2 · The frozen benchmark
You generated this by running `DS23_Module4_Benchmark.ipynb`. Load the locked number.
Your pipeline will be measured against it. Do NOT recompute it here.

In [ ]:
with open(DATA_PATH + "module4_benchmark.json") as f:
    bench = json.load(f)
print("frozen benchmark roc_auc:", bench["roc_auc"])
print("protocol:", bench["cv"], "| metric:", bench["metric"])
# If this cell errors, run DS23_Module4_Benchmark.ipynb first to create the JSON.


## 3 · Build the modeling table
The target is a negative review (score 1 or 2). This definition is fixed for everyone,
so results are comparable. Keep the SAME row set and the SAME evaluation protocol as the
benchmark, or the comparison is not fair.

In [ ]:
base = (items
        .merge(products, on="product_id", how="left")
        .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
        .merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id", how="left"))
base = base.dropna(subset=["review_score"]).copy()
base["neg_review"] = (base["review_score"] <= 2).astype(int)
base["order_purchase_timestamp"] = pd.to_datetime(base["order_purchase_timestamp"])
base = base.sort_values("order_purchase_timestamp").reset_index(drop=True)
model_df = base.dropna(subset=["product_category_name"]).copy()
y = model_df["neg_review"]
print("modeling table:", model_df.shape, "| negative rate:", round(y.mean(), 4))


## 4 · The frozen evaluation protocol
Use these exact objects for every comparison. Same cv, same metric as the benchmark.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)   # identical to the benchmark
METRIC = "roc_auc"

def evaluate(pipe, X):
    return cross_val_score(pipe, X, y, cv=cv, scoring=METRIC).mean()


---
# Your work starts here
Each section has a TODO. Fill it, and record your decision and reasoning in the
justification template (section 2 of the assignment document). One sentence of code is
not enough; the WHY is what is graded.

## 5 · Profile and handle missing values
Profile the gaps first, classify them (MCAR / MAR / MNAR), then choose an imputation
strategy. It must live inside the Pipeline.

In [ ]:
# TODO 5: profile missingness (counts, percentages, pattern), classify, and decide.
# Record: which strategy, why, which alternative you rejected, and the evidence.


## 6 · Outliers
Detect outliers (IQR, Z-score, or a multivariate method), then decide: remove, clip,
transform, or keep. Justify statistically AND in business terms.

In [ ]:
# TODO 6: detect and treat outliers. Remember: the decision must be justified,
# not automatic. Removing a real B2B whale is a mistake.


## 7 · Assemble the leakage-safe Pipeline
Build a ColumnTransformer (numeric route + categorical route) and wrap it with a model.
Everything that learns from data goes inside. This skeleton is a starting point; change
the strategies to the ones YOU justified above.

In [ ]:
num_cols = [c for c in model_df.columns if model_df[c].dtype != "object"
            and c not in ["neg_review", "review_score"]]
cat_col = "product_category_name"

# TODO 7: replace the placeholder strategies with your justified choices.
numeric = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # your choice + justification
    ("scale", StandardScaler()),                    # needed? justify per your model
])
categorical = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
pre = ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, [cat_col])])
pipe = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=1000))])


## 8 · Handle class imbalance
Only about one in six reviews is negative, and those are the ones that matter. Choose a
strategy (class_weight, resampling via imblearn Pipeline, or threshold tuning) and the
right metric. accuracy is not the right metric here; explain why.

In [ ]:
# TODO 8: address imbalance and choose your metric. If you resample, it MUST be inside
# an imblearn Pipeline so it runs on train folds only.


## 9 · Measure your lift against the benchmark
Same cv, same metric. Report your score, the benchmark, and the lift, honestly.

In [ ]:
X = model_df[num_cols + [cat_col]]
my_score = evaluate(pipe, X)
lift = my_score - bench["roc_auc"]
print(f"benchmark : {bench['roc_auc']:.4f}")
print(f"my score  : {my_score:.4f}")
print(f"my lift   : {lift:+.4f}")
# A small or negative lift is a valid result. Explain what it tells you.


## 10 · Own-data track only: your own baseline
If you chose the own-data track, you have no external benchmark. Build a minimal
baseline (a simple leakage-safe pipeline on your raw features), lock its score, and
measure the lift of your cleaned and engineered pipeline against it. Same idea, same
discipline.

In [ ]:
# TODO 10 (own-data track): baseline score, then your improved score, then the lift.


---
## Before you submit
- Every learning step is inside the Pipeline (no leakage). Check twice.
- The justification template is filled for every decision: what, why, alternative
  rejected, evidence.
- Your lift is reported honestly, whatever it is.
- The notebook runs top to bottom without errors.

Good luck. The reasoning is the point.